In [1]:
import sagemaker
from sagemaker.processing import ScriptProcessor, ProcessingInput, ProcessingOutput

session = sagemaker.Session()
role = sagemaker.get_execution_role()

bucket = session.default_bucket()
region = session.boto_region_name

print("Bucket:", bucket)
print("Region:", region)
print("Role:", role)

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
Bucket: sagemaker-us-east-1-864475311845
Region: us-east-1
Role: arn:aws:iam::864475311845:role/SageMakerStudioExecutionRole2026


In [2]:
image_uri = "864475311845.dkr.ecr.us-east-1.amazonaws.com/future-sales-processing:latest"
print(image_uri)

864475311845.dkr.ecr.us-east-1.amazonaws.com/future-sales-processing:latest


In [3]:
raw_prefix = "future-sales/processing/input"
output_prefix = "future-sales/processing/output"

raw_s3_uri = f"s3://{bucket}/{raw_prefix}"
output_s3_uri = f"s3://{bucket}/{output_prefix}"

print("Raw S3 URI:", raw_s3_uri)
print("Output S3 URI:", output_s3_uri)

Raw S3 URI: s3://sagemaker-us-east-1-864475311845/future-sales/processing/input
Output S3 URI: s3://sagemaker-us-east-1-864475311845/future-sales/processing/output


In [4]:
from pathlib import Path

data_dir = Path("../../data")
print("Data dir exists:", data_dir.exists())

for path in sorted(data_dir.rglob("*")):
    if path.is_file():
        print(path)

Data dir exists: True
../../data/inference/.gitkeep
../../data/predictions/.gitkeep
../../data/prep/.gitkeep
../../data/prep/features_train.csv.gz
../../data/raw/.gitkeep


In [5]:
import boto3
from pathlib import Path

s3 = boto3.client("s3")

local_file = Path("../../data/prep/features_train.csv.gz")

s3_key = f"{raw_prefix}/features_train.csv.gz"

print("Uploading:", local_file)
print("To:", f"s3://{bucket}/{s3_key}")

s3.upload_file(str(local_file), bucket, s3_key)

print("Upload complete")

Uploading: ../../data/prep/features_train.csv.gz
To: s3://sagemaker-us-east-1-864475311845/future-sales/processing/input/features_train.csv.gz
Upload complete


In [6]:
processor = ScriptProcessor(
    image_uri=image_uri,
    command=["python3"],
    role=role,
    instance_count=1,
    instance_type="ml.m5.large",
    sagemaker_session=session,
)

processor.run(
    code="../src/preprocess.py",
    inputs=[
        ProcessingInput(
            source=raw_s3_uri,
            destination="/opt/ml/processing/input",
        )
    ],
    outputs=[
        ProcessingOutput(
            source="/opt/ml/processing/output",
            destination=output_s3_uri,
        )
    ],
    wait=True,
    logs=True,
)

INFO:sagemaker:Creating processing-job with name future-sales-processing-2026-03-15-23-46-56-488


.......2026-03-15 23:48:06,797 - INFO - Cargando archivo desde /opt/ml/processing/input/features_train.csv.gz
2026-03-15 23:48:12,232 - INFO - Shape original: (10913850, 8)
2026-03-15 23:48:12,232 - INFO - Construyendo particiones
2026-03-15 23:48:12,232 - INFO - Variable objetivo detectada: item_cnt_month
2026-03-15 23:48:15,338 - INFO - train shape: (7639695, 8)
2026-03-15 23:48:15,338 - INFO - validation shape: (1637077, 8)
2026-03-15 23:48:15,338 - INFO - test shape: (1637078, 8)
2026-03-15 23:48:15,338 - INFO - submission_features shape: (1637078, 7)
2026-03-15 23:48:52,114 - INFO - Archivos guardados correctamente:
2026-03-15 23:48:52,114 - INFO -  - /opt/ml/processing/output/train.csv
2026-03-15 23:48:52,114 - INFO -  - /opt/ml/processing/output/validation.csv
2026-03-15 23:48:52,114 - INFO -  - /opt/ml/processing/output/test.csv
2026-03-15 23:48:52,115 - INFO -  - /opt/ml/processing/output/submission_features.csv
2026-03-15 23:48:52,115 - INFO - Preprocessing terminado exitosam

In [8]:
import boto3

s3 = boto3.client("s3")
response = s3.list_objects_v2(Bucket=bucket, Prefix=output_prefix)

for obj in response.get("Contents", []):
    print(obj["Key"])

future-sales/processing/output/submission_features.csv
future-sales/processing/output/test.csv
future-sales/processing/output/train.csv
future-sales/processing/output/validation.csv


In [9]:
import pandas as pd

for name in ["train.csv", "validation.csv", "test.csv", "submission_features.csv"]:
    print(f"\nPreview de {name}")
    df_preview = pd.read_csv(f"s3://{bucket}/{output_prefix}/{name}", nrows=5)
    display(df_preview)


Preview de train.csv


,date_block_num,shop_id,item_id,item_cnt_month,item_category_id,item_cnt_month_lag_1,item_cnt_month_lag_2,item_cnt_month_lag_3
0,5,22,19545,0.0,40,0.0,0.0,0.0
1,32,44,2866,0.0,25,0.0,0.0,1.0
2,7,55,9936,0.0,37,0.0,0.0,0.0
3,26,56,8462,0.0,43,0.0,0.0,0.0
4,21,57,17320,0.0,40,0.0,0.0,1.0



Preview de validation.csv


,date_block_num,shop_id,item_id,item_cnt_month,item_category_id,item_cnt_month_lag_1,item_cnt_month_lag_2,item_cnt_month_lag_3
0,23,44,3800,0.0,55,0.0,0.0,0.0
1,26,14,5900,0.0,30,1.0,0.0,1.0
2,9,57,152,0.0,45,0.0,0.0,0.0
3,2,13,6809,0.0,58,0.0,0.0,0.0
4,15,7,10876,0.0,37,0.0,0.0,0.0



Preview de test.csv


,date_block_num,shop_id,item_id,item_cnt_month,item_category_id,item_cnt_month_lag_1,item_cnt_month_lag_2,item_cnt_month_lag_3
0,24,33,9120,0.0,54,0.0,0.0,0.0
1,13,25,14279,0.0,58,1.0,0.0,0.0
2,5,3,20526,0.0,72,0.0,0.0,0.0
3,26,57,2883,0.0,25,2.0,0.0,0.0
4,8,17,17431,0.0,40,0.0,0.0,0.0



Preview de submission_features.csv


,date_block_num,shop_id,item_id,item_category_id,item_cnt_month_lag_1,item_cnt_month_lag_2,item_cnt_month_lag_3
0,24,33,9120,54,0.0,0.0,0.0
1,13,25,14279,58,1.0,0.0,0.0
2,5,3,20526,72,0.0,0.0,0.0
3,26,57,2883,25,2.0,0.0,0.0
4,8,17,17431,40,0.0,0.0,0.0
